# SIRIUS Stage B — Pass 5 recipe on Qwen2.5-Coder-1.5B

Frozen in `docs/16-stage-b-prereg.md`. Executes only after Stage A gates pass.
TPU v5e-8. No ship change until all 8 gates pass.


In [ ]:
!pip -q install transformers==4.46.0 peft==0.13.0 datasets==3.1.0 accelerate==1.1.1 sentencepiece
!git clone -q https://github.com/tayyabahmadq/Sirius.git /kaggle/working/sirius_repo
import torch, os, json
import torch_xla.core.xla_model as xm
dev = xm.xla_device()
print('TPU device:', dev, '| cores:', xm.xrt_world_size(), '| torch:', torch.__version__)
print('data rows:', sum(1 for _ in open('/kaggle/working/sirius_repo/data/sirius_train_pass5.jsonl', encoding='utf-8')))


In [ ]:
import torch, re, json, math, time, random, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset, DataLoader
import torch_xla.core.xla_model as xm

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
BASE='Qwen/Qwen2.5-Coder-1.5B-Instruct'
DEV=xm.xla_device()
tok=AutoTokenizer.from_pretrained(BASE)
tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16).to(DEV)
model.config.use_cache=False

# Frozen Pass 5 LoRA config: r=16 a=16 rsLoRA, q/k/v/o/down, layers 4-19
layers='|'.join(str(i) for i in range(4,20))
pat=rf'.*layers\.({layers})\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.down_proj)'
cfg=LoraConfig(r=16, lora_alpha=16, use_rslora=True, lora_dropout=0.05,
               target_modules=pat, bias='none', task_type='CAUSAL_LM')
model=get_peft_model(model, cfg)

# SplitLR: unfreeze base weights in last 2 layers at 1e-6; LoRA at 3e-5
slow,fast=[],[]
for n,p in model.named_parameters():
    if not p.requires_grad: continue
    if re.search(r'layers\.(22|23)\.', n): slow.append(p)
    else: fast.append(p)
opt=torch.optim.AdamW([{'params':slow,'lr':1e-6},{'params':fast,'lr':3e-5}],
                      weight_decay=0.01)
model.print_trainable_parameters()
print('split params: slow',sum(p.numel() for p in slow),'fast',sum(p.numel() for p in fast))


In [ ]:
# Data: pass5 LABEL-first corpus, completion-only loss, 512 tokens
rows=[json.loads(l) for l in open('/kaggle/working/sirius_repo/data/sirius_train_pass5.jsonl',encoding='utf-8') if l.strip()]
random.shuffle(rows)
MAXLEN=512
def encode(item):
    msgs=item['messages']
    prompt=tok.apply_chat_template(msgs[:-1],tokenize=False,add_generation_prompt=True)
    full=tok.apply_chat_template(msgs,tokenize=False)
    pi=tok(prompt,add_special_tokens=False)['input_ids']
    fi=tok(full,add_special_tokens=False)['input_ids'][:MAXLEN]
    labels=[-100]*min(len(pi),len(fi))+fi[len(pi):]
    return fi,labels
enc=[encode(r) for r in rows[:6000]]
class DS(Dataset):
    def __len__(self): return len(enc)
    def __getitem__(self,i): return enc[i]
def collate(batch):
    m=max(len(x[0]) for x in batch)
    ids=torch.full((len(batch),m),tok.eos_token_id,dtype=torch.long)
    lab=torch.full((len(batch),m),-100,dtype=torch.long)
    att=torch.zeros((len(batch),m),dtype=torch.long)
    for j,(a,b) in enumerate(batch):
        ids[j,:len(a)]=torch.tensor(a); lab[j,:len(b)]=torch.tensor(b); att[j,:len(a)]=1
    return ids,att,lab
loader=DataLoader(DS(),batch_size=4,shuffle=True,collate_fn=collate,num_workers=2)
print('train batches',len(loader))


In [ ]:
# Frozen Pass 5 optimisation: 2 epochs, cosine, 5% warmup, grad accum 4 (effective 16)
steps=2*math.ceil(len(loader)/4)
sched=get_cosine_schedule_with_warmup(opt,int(steps*0.05),steps)
model.train(); t0=time.time(); run=0.0; micro=0
for ep in range(2):
    for i,(ids,att,lab) in enumerate(loader):
        ids,att,lab=ids.to(DEV),att.to(DEV),lab.to(DEV)
        out=model(input_ids=ids,attention_mask=att,labels=lab)
        (out.loss/4).backward(); run+=out.loss.item(); micro+=1
        if (i+1)%4==0 or i==len(loader)-1:
            xm.optimizer_step(opt); sched.step(); opt.zero_grad(); xm.mark_step()
        if micro%100==0:
            print('step',micro,'loss',round(run/micro,4),'elapsed',round(time.time()-t0,1),flush=True)
    print('EPOCH',ep+1,'mean loss',round(run/micro,4),flush=True)
xm.mark_step()
model.save_pretrained('/kaggle/working/adapter_pass5_coder15')
print('ADAPTER SAVED', os.listdir('/kaggle/working/adapter_pass5_coder15'))
